In [0]:
import pyspark.sql.functions as F

##dim_date

In [0]:
df_dates = spark.table("silver.access_logs").select("date").distinct()

In [0]:
dim_date = df_dates.withColumn("day_name", F.date_format("date", "EEEE")) \
                   .withColumn("month_name", F.date_format("date", "MMMM")) \
                   .withColumn("day_of_week", F.dayofweek("date")) \
                   .withColumn("year", F.year("date")) \
                   .withColumn("is_weekend", F.when(F.col("day_of_week").isin(1, 7), True).otherwise(False))
                   
dim_date.write.format("delta").mode("overwrite").saveAsTable("gold.dim_date")

In [0]:
%sql
select * from gold.dim_date

##dim_clients

In [0]:
dim_clients = spark.table("silver.client_lookup")
dim_clients.write.format("delta").mode("overwrite").saveAsTable("gold.dim_clients")

##fact_web_traffic

In [0]:
fact_web_traffic = spark.table("silver.access_logs").select(
    "ip", 
    "date",
    "timestamp", 
    "method", 
    "endpoint", 
    "status_code", 
    "content_size"
)

fact_web_traffic.write.format("delta") \
                .mode("overwrite") \
                .partitionBy("date") \
                .saveAsTable("gold.fact_web_traffic")


In [0]:
%sql
SELECT * FROM gold.fact_web_traffic

##Golden_view

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW gold.vw_devops_dashboard AS
SELECT 
    f.timestamp,
    f.method,
    f.endpoint,
    f.status_code,
    f.content_size,
    c.hostname,
    d.day_name,
    d.month_name,
    d.is_weekend
FROM gold.fact_web_traffic f
LEFT JOIN gold.dim_clients c ON f.ip = c.ip
LEFT JOIN gold.dim_date d ON f.date = d.date
""")